---
title: Extreme Model Interpretation Computations
date: 11/2025
authors:
  - name: James Butler
    affiliations: ucb
  - name: Michelle Maclennan
    affiliation: bas
  - name: Fernando Pérez
    affiliation: ucb
  - name: Jon McAuliffe
    affiliation: ucb
affiliations:
  - id: ucb
    institution: University of California Berkeley
    ror: https://ror.org/01an7q238
    department: Statistics
  - id: bas
    institution: British Antarctic Survey
    ror: https://ror.org/01rhff309
---

The `gbex` model used for fitting the extreme conditional quantiles is a bit slower to run than the XGBoost models in the mean modelling section. As such, we will first compute all of the quantities necessary to interpret the fitted `gbex` models, saving the results as intermediate datasets that will be plotted and interpreted in a separate notebook (`extreme_modelling_results.ipynb`). That way, we don't have to rerun these relatively expensive operations each time we restart our Jupyter session.

This notebook is fully written in `R`, using the `extreme_antarctic_ars_R` conda environment (constructed from `environment_R.yml`).

:::{attention}
We recommend running this notebook in the background on a Jupyter server, as it may take a few hours to generate all of the necessary plotting datasets. To facilitate tracking the execution progress, we output regular checkpoints to log files, one for each of the snowfall and temperature models (`snow_gbex_interpretation_logs.txt`, `temp_gbex_interpretation_logs.txt`). From the project root directory, these logs can be found in the `/outputs/logs/` directory.
:::

In [1]:
# load R packages
library(ggplot2)
library(readr)
library(dplyr)
library(grf)
library(foreach)
library(doSNOW)
library(parallel)
library(jsonlite)
library(kernelshap)
library(shapviz)
pacman::p_load_gh('JVelthoen/gbex')
library(gbex)
source('~/extreme_antarctic_ARs/scripts/model_fitting/gbex/interpretation_utils_gbex.R')


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: iterators

Loading required package: snow


Attaching package: ‘parallel’


The following objects are masked from ‘package:snow’:

    clusterApply, clusterApplyLB, clusterCall, clusterEvalQ,
    clusterExport, clusterMap, clusterSplit, makeCluster, parApply,
    parCapply, parLapply, parRapply, parSapply, splitIndices,
    stopCluster




In [2]:
# our output directory
output_dir <- '~/extreme_antarctic_ARs/outputs/model_fitting/gbex/plotting_data/'

In [3]:
train_dat_full <- read_csv('~/extreme_antarctic_ARs/outputs/data_products/train.csv')
test_dat_full <- read_csv('~/extreme_antarctic_ARs/outputs/data_products/test.csv')

feature_cols <- c('cumulative_landfalling_area', 'max_south_extent', 
                  'max_IWV_ais', 'max_ocean_SLP_gradient', 
                  'max_landfalling_v850hPa', 'avg_landfalling_minomega')
X_train <- train_dat_full %>% select(feature_cols)
y_train_snow <- train_dat_full %>% select('cumulative_snowfall_ais') %>% pull()
y_train_temp <- train_dat_full %>% select('max_T2M_anomaly_ais') %>% pull()

X_test <- test_dat_full %>% select(feature_cols)
y_test_snow <- test_dat_full %>% select('cumulative_snowfall_ais') %>% pull()
y_test_temp <- test_dat_full %>% select('max_T2M_anomaly_ais') %>% pull()

Rows: 2466 Columns: 16
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (2): Label, region
dbl (14): max_area, mean_landfalling_area, cumulative_landfalling_area, dura...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 616 Columns: 16
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (2): Label, region
dbl (14): max_area, mean_landfalling_area, cumulative_landfalling_area, dura...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Warning message:
“Using an external vector in selections was deprecated in tidyselect 1.1.0.
ℹ Please use `all_of()` or `any_of()` instead.
  # Was:
  data %>% select(feature_cols)

  # Now:
  data %>% select(all_of(feature_cols))

See <https://

## Model Fitting

We first fit the `gbex` models for both the snowfall and temperature outcomes, evaluating performance on the held-out test-set.

### Snowfall

Fitting the generalized random forest for the intermediate quantile estimates. We seek to estimate the $\tau = 0.7$ quantile of the outcome distribution, conditional on each set of covariates.

In [4]:
tau_0 <- 0.7
grf_param_path <- paste0('~/extreme_antarctic_ARs/auxiliary_files/hyperparam_dictionaries/gbex/best_snow_grf.json')
best_grf_params <- fromJSON(grf_param_path)

fit_snow_threshold <- quantile_forest(
  X_train, y_train_snow,
  quantiles = tau_0,
  num.trees = best_grf_params$num_trees,
  min.node.size = best_grf_params$min_node_size,
  sample.fraction = best_grf_params$sample_frac,
  alpha = best_grf_params$alpha,
  honesty = FALSE,
  seed = 54321
)

Now we predict the intermediate quantiles for all of our training observations, and only retain those observations that exceed their intermediate quantile estimates.

In [5]:
u_train <- predict(fit_snow_threshold, X_train, quantiles = tau_0)$predictions[,1]
diffs <- y_train_snow - u_train
z_train <- diffs[diffs > 0] # only take exceedances above the intermediate threshold
X_train_gbex <- X_train[diffs > 0, ] # take the corresponding covariates as well
write_csv(X_train_gbex, paste0(output_dir, 'train_test_data/gbex_snow_trainX.csv'))
write_csv(data.frame(z_train), file = paste0(output_dir, 'train_test_data/gbex_snow_trainZ.csv'))

Let's fit the gbex model on these exceedances.

In [6]:
best_gbex_params <- fromJSON('~/extreme_antarctic_ARs/auxiliary_files/hyperparam_dictionaries/gbex/best_snow_gbex.json')

set.seed(12345)

gbex_snow <- gbex(y=z_train,
         X=X_train_gbex, 
         B=best_gbex_params$num_trees,
         lambda_scale=best_gbex_params$lambda_scale,
         lambda_ratio=best_gbex_params$lambda_ratio,
         depth=c(best_gbex_params$depth_sigma, best_gbex_params$depth_gamma),
         min_leaf_size=c(best_gbex_params$min_leaf_sigma, best_gbex_params$min_leaf_gamma),
         sf=best_gbex_params$sf)

Fit gbex
  |============================================================================| 100%


Now, let's apply our fitted intermediate quantile model to find the exceedances past the intermediate threshold.

In [7]:
u_test <- predict(fit_snow_threshold, X_test, quantiles = tau_0)$predictions[,1]
diffs <- y_test_snow - u_test
z_test <- diffs[diffs > 0] # only take exceedances above the intermediate threshold
X_test_gbex <- X_test[diffs > 0, ] # take the corresponding covariates as well

And we can compute the error on the training and test set by way of the GPD deviance.

In [10]:
train_preds <- predict(gbex_snow, X_train_gbex)
train_preds$z_train <- z_train
mean(apply(train_preds, 1, GP_dev))

[1] -1.930981

In [8]:
test_preds <- predict(gbex_snow, X_test_gbex)
test_preds$z_test <- z_test
mean(apply(test_preds, 1, GP_dev))

[1] -1.344778

We'll also save the features for all of the training and testing data used in the `gbex` procedure, to be used for model interpretations and diagnostics.

In [ ]:
# concatenate training and testing data to evaluate model diagnostics
X_full_gbex <- rbind(X_train_gbex, X_test_gbex)
z_full_gbex <- c(z_train, z_test)

write_csv(X_full_gbex, paste0(output_dir, 'train_test_data/gbex_snow_fullX.csv'))
write_csv(X_test_gbex, paste0(output_dir, 'train_test_data/gbex_snow_testX.csv'))
write_csv(data.frame(z_test), file = paste0(output_dir, 'train_test_data/gbex_snow_testZ.csv'))
write_csv(data.frame(z_full_gbex), file = paste0(output_dir, 'train_test_data/gbex_snow_fullZ.csv'))

### Temperature

We run through the same procedure as before, but for the temperature outcome.

In [12]:
tau_0 <- 0.7
grf_param_path <- paste0('~/extreme_antarctic_ARs/auxiliary_files/hyperparam_dictionaries/gbex/best_temp_grf.json')
best_grf_params <- fromJSON(grf_param_path)

fit_temp_threshold <- quantile_forest(
  X_train, y_train_temp,
  quantiles = tau_0,
  num.trees = best_grf_params$num_trees,
  min.node.size = best_grf_params$min_node_size,
  sample.fraction = best_grf_params$sample_frac,
  alpha = best_grf_params$alpha,
  honesty = FALSE,
  seed = 54321
)

Now we predict the intermediate quantiles for all of our training observations, and only retain those observations that exceed their intermediate quantile estimates.

In [13]:
u_train <- predict(fit_temp_threshold, X_train, quantiles = tau_0)$predictions[,1]
diffs <- y_train_temp - u_train
z_train <- diffs[diffs > 0] # only take exceedances above the intermediate threshold
X_train_gbex <- X_train[diffs > 0, ] # take the corresponding covariates as well
write_csv(X_train_gbex, paste0(output_dir, 'train_test_data/gbex_temp_trainX.csv'))
write_csv(data.frame(z_train), file = paste0(output_dir, 'train_test_data/gbex_temp_trainZ.csv'))

Let's fit the gbex model on these exceedances.

In [14]:
best_gbex_params <- fromJSON('~/extreme_antarctic_ARs/auxiliary_files/hyperparam_dictionaries/gbex/best_temp_gbex.json')

set.seed(12345)

gbex_temp <- gbex(y=z_train,
         X=X_train_gbex, 
         B=best_gbex_params$num_trees,
         lambda_scale=best_gbex_params$lambda_scale,
         lambda_ratio=best_gbex_params$lambda_ratio,
         depth=c(best_gbex_params$depth_sigma, best_gbex_params$depth_gamma),
         min_leaf_size=c(best_gbex_params$min_leaf_sigma, best_gbex_params$min_leaf_gamma),
         sf=best_gbex_params$sf)

Fit gbex
  |                                                                            |   0%

Warning message in first_guess(y, gamma_positive):
“Unconditional estimate of gamma is negative (initial estimate is set to 0.01”


  |============================================================================| 100%


Now, let's apply our fitted intermediate quantile model to find the exceedances past the intermediate threshold.

In [15]:
u_test <- predict(fit_temp_threshold, X_test, quantiles = tau_0)$predictions[,1]
diffs <- y_test_temp - u_test
z_test <- diffs[diffs > 0] # only take exceedances above the intermediate threshold
X_test_gbex <- X_test[diffs > 0, ] # take the corresponding covariates as well

And we can compute the error on this training and test set by way of the GPD deviance.

In [16]:
train_preds <- predict(gbex_temp, X_train_gbex)
train_preds$z_train <- z_train
mean(apply(train_preds, 1, GP_dev))

[1] 1.790998

In [17]:
test_preds <- predict(gbex_temp, X_test_gbex)
test_preds$z_test <- z_test
mean(apply(test_preds, 1, GP_dev))

[1] 2.415109

We'll also save the features for all of the training and testing data used in the `gbex` procedure, to be used for model interpretations and diagnostics.

In [ ]:
# concatenate training and testing data to evaluate model diagnostics
X_full_gbex <- rbind(X_train_gbex, X_test_gbex)
z_full_gbex <- c(z_train, z_test)

write_csv(X_full_gbex, paste0(output_dir, 'train_test_data/gbex_temp_fullX.csv'))
write_csv(X_test_gbex, paste0(output_dir, 'train_test_data/gbex_temp_testX.csv'))
write_csv(data.frame(z_test), file = paste0(output_dir, 'train_test_data/gbex_temp_testZ.csv'))
write_csv(data.frame(z_full_gbex), file = paste0(output_dir, 'train_test_data/gbex_temp_fullZ.csv'))

## Computing Intermediate Plotting Datasets

Now, let's compute quantities that will help us understand the relationships that our models have learned.

:::{attention}
We recommend running this section in the background as it will take a few hours to run through everything.
:::

### Snowfall

To track the progress of the computations while the notebook runs in the background, we will write the progress to a log file. A separate log file will be created for the temperature outcomes as well.

In [ ]:
snow_log_file <- '~/extreme_antarctic_ARs/outputs/logs/snow_gbex_interpretation_logs.txt'

Let's also load up all of our input training and testing datasets.

In [ ]:
X_test_gbex <- read_csv(paste0(output_dir, 'train_test_data/gbex_snow_testX.csv'))
X_full_gbex <- read_csv(paste0(output_dir, 'train_test_data/gbex_snow_fullX.csv'))
X_train_gbex <- read_csv(paste0(output_dir, 'train_test_data/gbex_snow_trainX.csv'))

z_test <- read_csv(paste0(output_dir, 'train_test_data/gbex_snow_testZ.csv')) %>% pull()

We start with the permutation score as a notion of variable importance.

In [ ]:
cat(paste0("\n--- Permutation score computation started at: ", Sys.time(), " ---\n"), 
    file = snow_log_file, append = TRUE)

set.seed(12345)

# verbose set to FALSE to run quietly in the background
permutation_scores <- permutation_score(
  gbex_snow, 
  data.frame(X_test_gbex), 
  z_test, 
  n_reps=500, 
  verbose=FALSE
)

cat(paste0("--- Permutation score computation finished at: ", Sys.time(), " ---\n"), 
    file = snow_log_file, append = TRUE)

In [ ]:
# save results so we don't have to keep recomputing them!
write_csv(permutation_scores, paste0(output_dir, 'variable_importance/snow_gbex_VI.csv'))

Now, let's compute 1D partial dependence plots. Once again, since it takes a bit longer to run through this algorithm, we'll save all the PDP results in an intermediate file and load up that file to plot the results. By default, we evaluate the PDP in the extents provided, for 50 points.

In [ ]:
# the extreme quantile we wish to look at
tau <- 0.995
# the quantile used to clip extreme observations in feature space
q <- 0.1
# the column names we need
cols <- colnames(X_full_gbex)

This next cell should only take a few minutes to run.

In [ ]:
cat(paste0("--- Starting 1D PDP computations at: ", Sys.time(), " ---\n"), 
    file = snow_log_file, append = TRUE)

for (i in 1:length(cols)) {
  var_name <- cols[i]
  
  # compute the 1D PDP
  plt_df <- compute_quantile_PDP_1D(
    object = gbex_snow,
    tau = tau, 
    X_data = X_full_gbex, 
    var_name = var_name, 
    grid_points = 50, 
    verbose = FALSE
  )
  
  file_name <- paste0(var_name, "_995_snow.csv")
  full_path <- paste0(output_dir, 'pdp_1D/', file_name)
  
  # save the result
  write_csv(plt_df$PD_df, full_path)

  log_message <- paste0("[", Sys.time(), "] Successfully created and saved PDP for: ", var_name, "\n")
  cat(log_message, file = snow_log_file, append = TRUE)
}

cat(paste0("--- Finished all 1D PDP computations at: ", Sys.time(), " ---\n\n"), 
    file = snow_log_file, append = TRUE)

Finally, let's compute 2D PDPs. To speed up the computation, we will compute each PDP plot's data in parallel.

In [ ]:
cat(paste0("--- Starting 2D PDP computations at: ", Sys.time(), " ---\n"), 
    file = snow_log_file, append = TRUE)

pairs <- combn(cols, 2)

# initialize the cluster
n_cores <- 5

cl <- makeCluster(n_cores, type='SOCK')
registerDoSNOW(cl)

# track progress of parallel loop
cat(paste0("\n--- Processing ", ncol(pairs), " variable pairs across ", n_cores, " cores ---\n"), 
    file = snow_log_file, append = TRUE)
progress <- function(n) {
  cat(paste0("[", Sys.time(), "] Completed ", n, " of ", ncol(pairs), " pairs...\n"),
    file = snow_log_file, append = TRUE)
}
opts <- list(progress = progress)

# run the parallel loop 
invisible({
  foreach(i = 1:ncol(pairs), .options.snow = opts, .packages = c("readr")) %dopar% {
    var1 <- pairs[1, i]
    var2 <- pairs[2, i]
    
    # compute the 2D PDP
    plt_df <- compute_quantile_PDP_2D(
      object = gbex_snow, 
      tau = tau, 
      X_data = X_full_gbex, 
      var_names = c(var1, var2), 
      grid_points = 50,
      verbose = FALSE 
    )
    
    # construct the filename and save
    file_name <- paste0(var1, "_", var2, "_995_snow.csv")
    full_path <- paste0(output_dir, 'pdp_2D/', file_name)
    write_csv(plt_df$PD_df, full_path)
    
    NULL # return NULL to keep memory overhead low
  }
})

# clean up and close the cluster
stopCluster(cl)
cat(paste0("\n--- All 2D PDP computations finished at: ", Sys.time(), " ---\n"),
   file = snow_log_file, append = TRUE)

Finally, we approximate SHAP values using the KernelSHAP method, which is flexible and model agnostic, allowing us to use a complex and specialized model like `gbex`. We will compute it using the exact algorithm since our feature space is relatively small, and use our training data as the background dataset.

In [ ]:
# define the prediction function, to input into kernelshap
pfun <- function(model, newdata) {
  preds <- predict(model, as.data.frame(newdata), what = 'quant', probs = tau)
  return(as.numeric(preds))
}

# log the start of the computation
cat(sprintf('[%s] Starting exact KernelSHAP computation...\n', Sys.time()), 
    file = snow_log_file, append = TRUE)

# compute SHAP values
ks <- kernelshap(object = gbex_snow, X = X_test_gbex, bg_X = X_train_gbex, pred_fun = pfun, verbose = FALSE)

# log that computation finished
cat(sprintf('[%s] KernelSHAP computation finished!\n', Sys.time()), 
    file = snow_log_file, append = TRUE)

# save the SHAP values matrix and R object for intermediate plotting
write_csv(as.data.frame(ks$S), paste0(output_dir, 'shap/snow_shap_values.csv'))
saveRDS(ks, paste0(output_dir, 'shap/snow_shap_object.rds'))

# log the successful save
cat(sprintf("[%s] SHAP values successfully saved to 'snow_shap_values.csv'\n", Sys.time()), file = snow_log_file, append = TRUE)

### Temperature

We run through the same procedure as in the previous section.

In [ ]:
temp_log_file <- '~/extreme_antarctic_ARs/outputs/logs/temp_gbex_interpretation_logs.txt'

In [ ]:
X_test_gbex <- read_csv(paste0(output_dir, 'train_test_data/gbex_temp_testX.csv'))
X_full_gbex <- read_csv(paste0(output_dir, 'train_test_data/gbex_temp_fullX.csv'))
X_train_gbex <- read_csv(paste0(output_dir, 'train_test_data/gbex_temp_trainX.csv'))

z_test <- read_csv(paste0(output_dir, 'train_test_data/gbex_temp_testZ.csv')) %>% pull()

Let's start with the permutation score as a notion of variable importance. This line will probably take about 20-30 mintues to run.

In [ ]:
cat(paste0("\n--- Permutation score computation started at: ", Sys.time(), " ---\n"), 
    file = temp_log_file, append = TRUE)

set.seed(12345)

# verbose set to FALSE to run quietly in the background
permutation_scores <- permutation_score(
  gbex_temp, 
  data.frame(X_test_gbex), 
  z_test, 
  n_reps=500, 
  verbose=FALSE
)

cat(paste0("--- Permutation score computation finished at: ", Sys.time(), " ---\n"), 
    file = temp_log_file, append = TRUE)

In [ ]:
# save results so we don't have to keep recomputing them!
write_csv(permutation_scores, paste0(output_dir, 'variable_importance/temp_gbex_VI.csv'))

Now, let's compute 1D partial dependence plots. Once again, since it takes a bit longer to run through this algorithm, we'll save all the PDP results in an intermediate file and load up that file to plot the results. By default, we evaluate the PDP in the extents provided, for 50 points.

In [ ]:
# the extreme quantile we wish to look at
tau <- 0.995
# the quantile used to clip extreme observations in feature space
q <- 0.1
# the column names we need
cols <- colnames(X_full_gbex)

This next cell should only take a few minutes to run.

In [ ]:
cat(paste0("--- Starting 1D PDP computations at: ", Sys.time(), " ---\n"), 
    file = temp_log_file, append = TRUE)

for (i in 1:length(cols)) {
  var_name <- cols[i]
  
  # compute the 1D PDP
  plt_df <- compute_quantile_PDP_1D(
    object = gbex_temp,
    tau = tau, 
    X_data = X_full_gbex, 
    var_name = var_name, 
    grid_points = 50, 
    verbose = FALSE
  )
  
  file_name <- paste0(var_name, "_995_temp.csv")
  full_path <- paste0(output_dir, 'pdp_1D/', file_name)
  
  # save the result
  write_csv(plt_df$PD_df, full_path)

  log_message <- paste0("[", Sys.time(), "] Successfully created and saved PDP for: ", var_name, "\n")
    cat(log_message, file = temp_log_file, append = TRUE)
}

cat(paste0("--- Finished all 1D PDP computations at: ", Sys.time(), " ---\n\n"), 
    file = temp_log_file, append = TRUE)

Finally, let's compute 2D PDPs. To speed up the computation, we will compute each PDP plot's data in parallel.

In [ ]:
pairs <- combn(cols, 2)

# initialize the cluster
n_cores <- 5

cl <- makeCluster(n_cores, type='SOCK')
registerDoSNOW(cl)

# track progress of parallel loop
cat(paste0("\n--- Processing ", ncol(pairs), " variable pairs across ", n_cores, " cores ---\n"), 
    file = temp_log_file, append = TRUE)
progress <- function(n) {
  cat(paste0("[", Sys.time(), "] Completed ", n, " of ", ncol(pairs), " pairs...\n"),
    file = temp_log_file, append = TRUE)
}
opts <- list(progress = progress)

# run the parallel loop 
invisible({
  foreach(i = 1:ncol(pairs), .options.snow = opts, .packages = c("readr")) %dopar% {
    var1 <- pairs[1, i]
    var2 <- pairs[2, i]
    
    # compute the 2D PDP
    plt_df <- compute_quantile_PDP_2D(
      object = gbex_temp, 
      tau = tau, 
      X_data = X_full_gbex, 
      var_names = c(var1, var2), 
      grid_points = 50,
      verbose = FALSE 
    )
    
    # construct the filename and save
    file_name <- paste0(var1, "_", var2, "_995_temp.csv")
    full_path <- paste0(output_dir, 'pdp_2D/', file_name)
    write_csv(plt_df$PD_df, full_path)
    
    NULL # return NULL to keep memory overhead low
  }
})

# clean up and close the cluster
stopCluster(cl)
cat(paste0("\n--- All 2D PDPs successfully processed at: ", Sys.time(), " ---\n"),
   file = temp_log_file, append = TRUE)

As before, we approximate SHAP values using the KernelSHAP method, which is flexible and model agnostic, allowing us to use a complex and specialized model like `gbex`. We'll save the shap values as an intermediate data product, and call them up in the results interpretation notebook.

In [ ]:
# define the prediction function, to input into kernelshap
pfun <- function(model, newdata) {
  preds <- predict(model, as.data.frame(newdata), what = 'quant', probs = tau)
  return(as.numeric(preds))
}

# log the start of the computation
cat(sprintf('[%s] Starting exact KernelSHAP computation...\n', Sys.time()), 
    file = temp_log_file, append = TRUE)

# compute SHAP values
ks <- kernelshap(object = gbex_temp, X = X_test_gbex, bg_X = X_train_gbex, pred_fun = pfun, verbose = FALSE)

# log that computation finished
cat(sprintf('[%s] KernelSHAP computation finished!\n', Sys.time()), 
    file = temp_log_file, append = TRUE)

# save the SHAP values matrix and R object for intermediate plotting
write_csv(as.data.frame(ks$S), paste0(output_dir, 'shap/temp_shap_values.csv'))
saveRDS(ks, paste0(output_dir, 'shap/temp_shap_object.rds'))

# log the successful save
cat(sprintf("[%s] SHAP values successfully saved to 'temp_shap_values.csv'\n", Sys.time()), file = temp_log_file, append = TRUE)